# #237 BJ-UNIFIED-POWER-FEATURE-SUPPORT-001 — causal power feature support

Rebuild timestamp-safe league/team/matchup strength features on the canonical
fixture identity. This notebook executes the reusable builder, the #180
support-loss waterfall reconstruction and support/calibration reporting.
It never opens 2024+ sealed labels for design (REQ-PW07).

In [1]:

from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "results").exists():
    ROOT = ROOT.parents[2]
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
import bj237_power_lib as lib
print("root:", ROOT)
print("pandas:", pd.__version__)


root: /home/drwho1985/beijing-lot-237
pandas: 3.0.3


## 1. Identity and coverage profiling (REQ-PW02, REQ-PW06)

In [2]:

matches = lib.load_xlsm_matches()
epochs = lib.load_decision_epochs(ROOT)
print("xlsm matches:", len(matches), "issues:", matches['issue'].nunique())
print("#203 epochs:", len(epochs), "issues")
print("blank team ids:", ((matches['home_team_id'].isna()) | (matches['home_team_id'].astype(str)=='')).sum())
print("dup issue+match_idx:", matches.duplicated(['issue','match_idx']).sum())
print("score present:", matches['fthg'].notna().sum())


xlsm matches: 195269 issues: 701
#203 epochs: 394 issues
blank team ids: 2566
dup issue+match_idx: 2559
score present: 189925


### 1a. Support-loss waterfall (REQ-PW06)

In [3]:

waterfall = pd.read_csv(ROOT / "results/bj237_power_feature_support_v1/support_loss_waterfall.csv")
display(waterfall)


,stage,label,rows,issues,reason
0,0.0,raw source rows,195269,701,NaN
1,1.0,identity ok (team ids + unique key),192703,701,NaN
2,2.0,league present,192703,701,NaN
3,3.0,decision epoch exists (#203),102243,344,NaN
4,NaN,NaN,2566,6,IDENTITY_MISSING_TEAM_ID
5,NaN,NaN,0,0,IDENTITY_DUPLICATE_ISSUE_MATCHIDX
6,NaN,NaN,0,0,IDENTITY_LEAGUE_BLANK
7,NaN,NaN,90460,357,NO_DECISION_EPOCH
8,NaN,NaN,1121,256,NOT_ELIGIBLE_HISTORY
9,NaN,NaN,101122,344,HISTORY_ELIGIBLE_SCORE_PRESENT


### 1b. #180 accepted P_POWER intersection independently reproduced

In [4]:

rec = pd.read_csv(ROOT / "results/bj237_power_feature_support_v1/ppower_intersection_reconstruction.csv")
display(rec)


,metric,value
0,cells_total,357705
1,issues_total,419
2,freeze_rows,1558
3,freeze_issues,189
4,freeze_complete_rows,1558
5,merged_power_cells,4674
6,line_matched_cells,4674
7,published_cells,234
8,published_issues,9
9,published_issues_list,"['20062', '21064', '21071', '22064', '23024', ..."


## 2. Causal feature builder output (REQ-PW01/03/04)

In [5]:

feat = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v1/power_features.parquet")
print("cells:", len(feat), "issues:", feat['issue'].nunique())
print("triplet_ok:", feat['triplet_ok'].sum(), "line_ok:", feat['line_ok'].sum(),
      "unseen:", feat['unseen_mask'].sum())
display(feat.head(6))


cells: 306720 issues: 344
triplet_ok: 306720 line_ok: 306720 unseen: 1203


,issue,match_idx,match_id,kickoff_ts_utc,decision_epoch_utc,year,league,home_team_id,away_team_id,native_handicap_line,line_ok,unseen_mask,min_history_n,home_history_n,away_history_n,triplet_ok,uncertainty,outcome_code,power_probability
0,190101,1,90465,2018-12-31 08:00:00+00:00,2018-12-31 08:00:00+00:00,2019,澳洲甲,2910,2913,1.0,True,False,127,132,127,True,0.024846,3,0.502418
1,190101,1,90465,2018-12-31 08:00:00+00:00,2018-12-31 08:00:00+00:00,2019,澳洲甲,2910,2913,1.0,True,False,127,132,127,True,0.024846,1,0.235119
2,190101,1,90465,2018-12-31 08:00:00+00:00,2018-12-31 08:00:00+00:00,2019,澳洲甲,2910,2913,1.0,True,False,127,132,127,True,0.024846,0,0.262463
3,190101,2,90466,2018-12-31 19:00:00+00:00,2018-12-31 08:00:00+00:00,2019,以超,1607,1659,0.0,True,False,128,149,128,True,0.024749,3,0.405932
4,190101,2,90466,2018-12-31 19:00:00+00:00,2018-12-31 08:00:00+00:00,2019,以超,1607,1659,0.0,True,False,128,149,128,True,0.024749,1,0.289509
5,190101,2,90466,2018-12-31 19:00:00+00:00,2018-12-31 08:00:00+00:00,2019,以超,1607,1659,0.0,True,False,128,149,128,True,0.024749,0,0.304559


### 2a. Publishable support by year (REQ-PW08)

In [6]:

pub = feat[feat['triplet_ok'] & feat['line_ok'] & ~feat['unseen_mask']]
by_year = pub.groupby('year').agg(cells=('outcome_code','count'),
                                  matches=('match_idx','nunique'),
                                  issues=('issue','nunique')).reset_index()
display(by_year)


,year,cells,matches,issues
0,2019,52833,680,55
1,2020,35844,572,44
2,2021,56427,632,58
3,2022,50214,570,58
4,2023,47127,503,55
5,2024,48552,523,59
6,2025,14520,547,15


### 2b. Diagnostics: history depth, uncertainty, Elo, form, rest

In [7]:

diag = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v1/power_feature_diagnostics.parquet")
display(diag[['issue','match_idx','league','home_team_id','away_team_id',
              'min_history_n','uncertainty','diag_home_elo','diag_away_elo',
              'diag_home_rest_hours','diag_home_matches_7d']].head(8))


,issue,match_idx,league,home_team_id,away_team_id,min_history_n,uncertainty,diag_home_elo,diag_away_elo,diag_home_rest_hours,diag_home_matches_7d
0,190101,1,澳洲甲,2910,2913,127,0.024846,1277.590306,1565.588200,191.000278,0
1,190101,2,以超,1607,1659,128,0.024749,1559.892796,1530.477223,82.500000,1
2,190101,3,澳洲甲,20394,14549,130,0.024558,1456.660991,1543.478527,116.166667,1
3,190101,4,英超,31,59,225,0.018667,1707.255306,1701.933221,38.000278,2
4,190101,5,英乙,1193,1199,104,0.027456,1425.373092,1519.583861,38.000278,2
5,190101,6,英超,19,29,243,0.017962,1880.281891,1582.142163,35.500000,2
6,190101,7,英冠,20,50,225,0.018667,1649.984100,1549.066500,38.000278,2
7,190101,8,英冠,21,18,209,0.019368,1540.001574,1623.920658,38.000278,2


## 3. Support and calibration by issue/year/league/line (REQ-PW07)

In [8]:

sup = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v1/support_summary.parquet")
print("support groups:", len(sup))
display(sup.head(12))


support groups: 3284


,year,league,native_handicap_line,rows,issues,matches
0,2019,世俱杯,-1.0,9,2,3
1,2019,世俱杯,0.0,6,1,2
2,2019,世俱杯,2.0,3,1,1
3,2019,世青杯,-3.0,3,1,1
4,2019,世青杯,-2.0,3,1,1
5,2019,世青杯,-1.0,15,3,5
6,2019,世青杯,0.0,54,5,17
7,2019,世青杯,1.0,3,1,1
8,2019,世青杯,2.0,12,3,4
9,2019,东亚女足,-3.0,3,1,1


In [9]:

cal = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v1/calibration_through_2023.parquet")
print("calibration groups (<=2023):", len(cal))
if len(cal):
    display(cal.head(12))
    cal_year = cal.groupby('year').agg(n=('n','sum'),
                                       mean_pred=('mean_pred','mean'),
                                       mean_actual=('mean_actual','mean')).reset_index()
    display(cal_year)


calibration groups (<=2023): 1899


,year,league,outcome_code,n,mean_pred,mean_actual
0,2019,世俱杯,0,6,0.324553,0.333333
1,2019,世俱杯,1,6,0.230200,0.333333
2,2019,世俱杯,3,6,0.445248,0.333333
3,2019,世青杯,0,47,0.367290,0.446809
4,2019,世青杯,1,47,0.245145,0.212766
5,2019,世青杯,3,47,0.387565,0.340426
6,2019,东亚女足,0,5,0.277249,0.200000
7,2019,东亚女足,1,5,0.174388,0.400000
8,2019,东亚女足,3,5,0.548363,0.400000
9,2019,东亚杯,0,6,0.463119,0.166667


,year,n,mean_pred,mean_actual
0,2019,52458,0.333333,0.333333
1,2020,35211,0.333333,0.333333
2,2021,55671,0.333333,0.333333
3,2022,49641,0.333333,0.333333
4,2023,46701,0.333333,0.333333


## 4. Internal vs external data contributions (REQ-PW05)

In [10]:

ext = pd.read_csv(ROOT / "results/bj237_power_feature_support_v1/external_football_data_census.csv")
print("external football-data files:", len(ext), "rows:", ext['rows'].sum())
display(ext)


external football-data files: 25 rows: 8982


,file,rows,competition,first_date,last_date,error
0,2021_D1.csv,306,D1,NaN,NaN,NaN
1,2021_E0.csv,380,E0,NaN,NaN,NaN
2,2021_F1.csv,380,F1,NaN,NaN,NaN
3,2021_I1.csv,380,I1,NaN,NaN,NaN
4,2021_SP1.csv,380,SP1,NaN,NaN,NaN
5,2122_D1.csv,306,D1,NaN,NaN,NaN
6,2122_E0.csv,380,E0,NaN,NaN,NaN
7,2122_F1.csv,380,F1,NaN,NaN,NaN
8,2122_I1.csv,380,I1,NaN,NaN,NaN
9,2122_SP1.csv,380,SP1,NaN,NaN,NaN


## 5. Sealed-label policy verification (REQ-PW07)

In [11]:

assert cal['year'].max() <= 2023, "calibration must never open 2024+"
print("OK: calibration table uses only <=2023 labels")


OK: calibration table uses only <=2023 labels


## 6. Package verification

In [12]:

import json
hashes = lib.sha256_file
sums = json.loads((ROOT / "results/bj237_power_feature_support_v1/SHA256SUMS.json").read_text())
import hashlib
mismatch = []
for name, expected in sums.items():
    p = ROOT / "results/bj237_power_feature_support_v1" / name
    if hashlib.sha256(p.read_bytes()).hexdigest() != expected:
        mismatch.append(name)
print("files:", len(sums), "mismatches:", mismatch if mismatch else "none")
manifest = json.loads((ROOT / "results/bj237_power_feature_support_v1/manifest.json").read_text())
print("base commit:", manifest.get("base_main_commit"))


files: 11 mismatches: none
base commit: 784a23badf38d51bb2aee6d7687479e425d1af80


## Conclusion (REQ-PW08)

Positive timestamp-safe power support: **305,517 causal publishable cells across
344 issues** (vs the accepted #180 coverage of 234 cells / 9 issues). The
#180 support-loss waterfall is independently reconstructed: identity loss
(2,566 rows missing team ids), league/candidate-pool gap (#180 only froze 189
of 702 issues from the #134/#179 candidate pools) and causal-cutoff/line
gates (kickoff >= epoch, native line grid, complete triplet) together explain
the nine-issue P_POWER intersection, which this rebuild expands to the full
#203 decision-epoch space.